In [1]:
%matplotlib qt
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation, PillowWriter
from matplotlib.gridspec import GridSpec
import numpy.ma as ma

In [4]:
# Load image and rot3 and rot6

img = np.load('data/STEM img.npy') 
rot3 = np.load('data/rot3 img.npy')
rot6 = np.load('data/rot6 img.npy')

# Create a mask
mask = np.ones(rot3.shape, dtype=bool) # Masking all

# Apply the mask to the data
masked_data = ma.array(rot3, mask=mask)

In [6]:
def compute_ij(step_size, mask_size):
    # Generate positions for updating symmetry image
    positions = []
    for i in range(mask_size):
        for j in range(0, mask_size, step_size):
            positions.append([i, j])
    return np.array(positions)

def update_mask(row_ind, col_ind, step_size, mask_size):
    mask = np.ones((mask_size, mask_size), dtype=bool)  # Masking all
    i = max(0, row_ind - 1)
    mask[0:i, :] = False
    j = min(col_ind + step_size, mask_size)
    mask[row_ind, 0:j] = False
    return mask

# Load image and rot3 (for testing, using random data)
img = np.load('data/STEM img.npy')
rot3 = np.load('data/rot3 img.npy')

# Parameter settings
mask_size = rot3.shape[0]
step_size = 120

# Create a mask (initially masking everything)
mask = np.ones_like(rot3, dtype=bool)
masked_data = ma.array(rot3, mask=mask)

fig, axes = plt.subplots(1, 2, figsize=(10, 5))
ax1, ax2 = axes.ravel()

im1 = ax1.imshow(img, cmap='viridis')
im2 = ax2.imshow(masked_data, cmap='viridis', vmin=rot3.min(), vmax=rot3.max())

# Compute positions for the sliding kernel
ijs = compute_ij(step_size, mask_size)

def update(frame):
    i, j = ijs[frame]
    mask = update_mask(i, j, step_size, mask_size)
    masked_data.mask = mask  # Update mask directly
    im2.set_array(masked_data)
    return [im2]

# Create the animation
ani = FuncAnimation(
    fig,
    update,
    frames=len(ijs),
    interval=5,  # Interval between frames in milliseconds
    blit=True,  # Blit=False to avoid rendering issues
)

## Save to gif

In [ ]:
# Save the animation as a GIF
ani.save(
    "symmetry_animation.gif",
    writer=PillowWriter(fps=200),
    savefig_kwargs={"transparent": True, "pad_inches": 0},
)

plt.close(fig)